# AudioGen Interactive GPU Server Notebook
Runs the FastAPI speech synthesis server, Cloudflare quick tunnel, and idle watchdog on Kaggle.

In [ ]:
# Cell 1: Dependencies & Environment Setup
import os
import sys
from pathlib import Path
import shutil

# Resolve repository root across various execution environments:
# 1. Local execution: notebook located in server/ or repository root
# 2. Remote Kaggle execution: notebook executed in /kaggle/working
repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/working/audiogen"),
    Path("/kaggle/working"),
]

for candidate in repo_candidates:
    if (candidate / "server").is_dir() and (candidate / "voices").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if (candidate / "src").is_dir() and str(candidate / "src") not in sys.path:
            sys.path.insert(0, str(candidate / "src"))
        req_file = candidate / "requirements.txt"
        if req_file.exists():
            print(f"Installing dependencies from {req_file}...")
            get_ipython().system(f"pip install -q -r {req_file}")
        break

# In remote Kaggle environment, if server/voices packages are not present locally:
if not any((Path(p) / "server").is_dir() and (Path(p) / "voices").is_dir() for p in sys.path):
    print("Remote Kaggle environment detected: cloning audiogen repository...")
    os.environ["GIT_TERMINAL_PROMPT"] = "0"
    get_ipython().system(
        "git clone https://github.com/lovishgoyal145/audiogen.git /kaggle/working/audiogen"
    )
    remote_repo = Path("/kaggle/working/audiogen")
    if remote_repo.is_dir():
        if str(remote_repo) not in sys.path:
            sys.path.insert(0, str(remote_repo))
        if (remote_repo / "src").is_dir() and str(remote_repo / "src") not in sys.path:
            sys.path.insert(0, str(remote_repo / "src"))
        req_file = remote_repo / "requirements.txt"
        if req_file.exists():
            print(f"Installing dependencies from {req_file}...")
            get_ipython().system(f"pip install -q -r {req_file}")

# Ensure cloudflared binary is available
if not shutil.which("cloudflared"):
    print("Installing cloudflared binary...")
    get_ipython().system("wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    get_ipython().system("chmod +x /tmp/cloudflared")
    get_ipython().system("mv /tmp/cloudflared /usr/local/bin/cloudflared 2>/dev/null || cp /tmp/cloudflared /usr/bin/cloudflared 2>/dev/null || true")



In [ ]:
# Cell 2: Load & Validate Runtime Secrets from Attached Kaggle Dataset
import os, sys, json
from pathlib import Path

REQUIRED_SECRETS = [
    "TUNNEL_REGISTRY_WEBHOOK_URL",
    "TUNNEL_REGISTRY_AUTH_TOKEN",
    "SERVER_BEARER_TOKEN",
]
OPTIONAL_SECRETS = [
    "HF_TOKEN",
]
ALL_SECRETS = REQUIRED_SECRETS + OPTIONAL_SECRETS

# Candidate paths for private dataset mounted under /kaggle/input/
dataset_candidates = [
    Path("/kaggle/input/audiogen-secrets"),
    Path("/kaggle/input"),
    Path.cwd() / "scratch" / "test_dataset",
]
if Path("/kaggle/input").is_dir():
    for sub in Path("/kaggle/input").iterdir():
        if sub.is_dir() and (sub / "secrets.json").is_file():
            dataset_candidates.insert(0, sub)

loaded_source = None
for candidate in dataset_candidates:
    if not candidate.exists():
        continue
    # 1. Try reading secrets.json
    secrets_json = candidate / "secrets.json" if candidate.is_dir() else candidate
    if secrets_json.is_file():
        try:
            with open(secrets_json, "r", encoding="utf-8") as f:
                data = json.load(f)
            for key in ALL_SECRETS:
                if key in data and str(data[key]).strip():
                    os.environ[key] = str(data[key]).strip()
            loaded_source = str(secrets_json)
            break
        except Exception as exc:
            print(f"Notice: Error reading {secrets_json}: {exc}")

    # 2. Try individual files in candidate directory
    if candidate.is_dir():
        found_any = False
        for key in ALL_SECRETS:
            kf = candidate / key
            if kf.is_file():
                try:
                    val = kf.read_text(encoding="utf-8").strip()
                    if val:
                        os.environ[key] = val
                        found_any = True
                except Exception:
                    pass
        if found_any:
            loaded_source = str(candidate)
            break

if loaded_source:
    print(f"Loaded runtime secrets from: {loaded_source}")

# Fallback to UserSecretsClient (interactive UI sessions only)
if not all(os.environ.get(k) for k in REQUIRED_SECRETS):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ALL_SECRETS:
            if not os.environ.get(key):
                val = secrets.get_secret(key)
                if val:
                    os.environ[key] = str(val).strip()
        print("Loaded remaining secrets from Kaggle UserSecretsClient.")
    except Exception as exc:
        print(f"Notice: UserSecretsClient unavailable ({exc}).")

# Hard Startup Assertion Gate
missing = [k for k in REQUIRED_SECRETS if not os.environ.get(k)]
print("")
print("--- Startup Secret Validation ---")
for key in ALL_SECRETS:
    status = "PRESENT" if os.environ.get(key) else "MISSING"
    print(f"{key}: {status}")
print("---------------------------------")

if missing:
    missing_str = ", ".join(missing)
    err_msg = f"FATAL: Missing required runtime secrets: {missing_str}"
    print(err_msg, file=sys.stderr)
    raise RuntimeError(err_msg)

if os.environ.get("HF_TOKEN"):
    try:
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"])
        print("Hugging Face Hub authentication successful via HF_TOKEN.")
    except Exception as exc:
        print(f"Notice: huggingface_hub login call: {exc}")

print("All runtime secrets verified. Proceeding with service startup.")


In [ ]:
# Cell 2b: Production Path & Secondary Standalone Diagnostic Probe
import sys
import traceback

print("==================================================")
print("=== [PHASE 1 DIAGNOSTIC] 1. PRODUCTION PATH    ===")
print("==================================================")
try:
    from audiogen.engine import Synthesizer
    print("Calling production Synthesizer()...")
    test_synth = Synthesizer()
    print("Production Synthesizer() initialized successfully:", test_synth)
except Exception as exc:
    print(f"Production Synthesizer() FAILED: {type(exc).__name__}: {exc}", file=sys.stderr)
    traceback.print_exc(file=sys.stderr)
    traceback.print_exc(file=sys.stdout)
    sys.stderr.flush()
    sys.stdout.flush()

print("\n==================================================")
print("=== [PHASE 1 DIAGNOSTIC] 2. STANDALONE PROBE   ===")
print("==================================================")
try:
    from transformers import AutoModel
    print("Calling AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True)...")
    test_model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True)
    print("Standalone AutoModel loaded successfully:", type(test_model))
except Exception as exc:
    print(f"Standalone AutoModel FAILED: {type(exc).__name__}: {exc}", file=sys.stderr)
    traceback.print_exc(file=sys.stderr)
    traceback.print_exc(file=sys.stdout)
    sys.stderr.flush()
    sys.stdout.flush()
print("==================================================")


In [ ]:
# Cell 3: Start FastAPI Speech Synthesis Server
import threading
import time
import uvicorn
from server.app import app

# Configure uvicorn for non-blocking execution on port 17000
config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=17000,
    log_level="info",
)
server = uvicorn.Server(config)
server_thread = threading.Thread(
    target=server.run,
    daemon=True,
    name="UvicornServerThread",
)
server_thread.start()

# Wait briefly for socket binding
time.sleep(1.5)
print(f"FastAPI server running in background thread on port 17000.")


In [ ]:
# Cell 4: Launch Cloudflare Tunnel & Publish URL
import os, sys
from server.tunnel import start_tunnel
from server.registry import publish_tunnel_url

tunnel, tunnel_url = start_tunnel(
    target_url="http://localhost:17000",
    startup_timeout_seconds=30.0,
)
print(f"Cloudflare tunnel online: {tunnel_url}")

token = (
    os.environ.get("SERVER_BEARER_TOKEN")
    or os.environ.get("SHARED_SECRET")
    or ""
).strip()
webhook_url = os.environ.get("TUNNEL_REGISTRY_WEBHOOK_URL")

try:
    result = publish_tunnel_url(tunnel_url=tunnel_url, secret=token)
    print(f"Successfully published tunnel URL to registry: {result}")
except Exception as exc:
    print(f"FATAL: Failed to publish tunnel URL to registry: {exc}", file=sys.stderr)
    raise RuntimeError(f"Tunnel publication failure: {exc}") from exc


In [ ]:
# Cell 5: Start Idle Watchdog Daemon
from server.watchdog import IdleWatchdog

# Initialize watchdog (default 600s timeout, touches on requests)
watchdog = IdleWatchdog(idle_timeout_seconds=600.0, check_interval_seconds=1.0)
app.state.watchdog = watchdog
watchdog.start()
print(f"Idle watchdog started (timeout: {watchdog.idle_timeout_seconds}s).")


In [ ]:
# Cell 6: Main Execution Loop
import time

print("AudioGen interactive GPU server is online and awaiting requests.")
print("Session will automatically shut down after 600s of inactivity.")

try:
    while True:
        time.sleep(1.0)
except KeyboardInterrupt:
    print("Server interrupted. Stopping...")
